# MIT–BIT charging forecast refinement and Discrete SAC

Compare 5-, 10-, and 15-cycle forecasts of delivered Wh and charging minutes on the same held-out batteries as Notebook 18. Then compare the original and improved MPC with Discrete SAC on the original 164 starting states. All three actions are observed charging rates and discharge remains fixed at 3C.

Run in the LifeGuard-RL project. Inputs are the Notebook 14, 16, 17, and 18 result folders or ZIPs and the extracted MIT–BIT v3 workbooks or official ZIP. The raw reader opens one workbook at a time. The notebook uses one CPU thread.

In [1]:
import os
import re
for name in ("OPENBLAS_NUM_THREADS", "OMP_NUM_THREADS", "MKL_NUM_THREADS",
             "VECLIB_MAXIMUM_THREADS", "BLIS_NUM_THREADS"):
    os.environ[name] = "1"

import json
from collections import defaultdict
from dataclasses import dataclass
from itertools import islice, product
from pathlib import Path, PurePosixPath
from contextlib import contextmanager
from tempfile import SpooledTemporaryFile
from xml.etree import ElementTree
from zipfile import ZIP_DEFLATED, ZipFile

import numpy as np
import pandas as pd
import torch
from torch import nn
from scipy.linalg import solve
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics.pairwise import euclidean_distances
from sklearn.linear_model import Ridge
from threadpoolctl import threadpool_limits

NOTEBOOK_14_EXPORT = None
NOTEBOOK_16_EXPORT = None
NOTEBOOK_17_EXPORT = None
NOTEBOOK_18_EXPORT = None
DATASET_PATH = None
OUTPUT_ROOT = None

BLOCK = 5
BLOCKS = 3
HORIZONS = tuple(range(1, BLOCKS + 1))
RATES = (1, 2, 3)
FOLDS = (1, 2, 3, 4, 5)
POLICY_STARTS = (11, 41, 66)
FIRST_START, LAST_START = 11, 91
ENERGY_TARGET = 0.90
OVERPREDICTION_QUANTILE = 0.90
TOLERANCE = 1e-9
BOOTSTRAP_DRAWS = 2000
SEED = 42
KEYS = ["cell_id", "start_cycle"]
POLICIES = {"fixed_1c": "Fixed 1C", "model": "Moving-target controller",
            "energy_budget": "Fixed-budget controller"}
OUTPUT_NAME = "lifeguard_mit_bit_charging_sac_results"
PILOT_NAME = "lifeguard_mit_bit_energy_budget_controller_results"

cwd = Path.cwd()
project = next((p for p in (cwd, *cwd.parents)
                if (p / "README_LifeGuard_RL.md").is_file() and (p / "reports").is_dir()), None)
if project is None:
    raise FileNotFoundError("Open this notebook from within the LifeGuard-RL project.")



def report_path(override, name):
    if override is not None:
        return Path(override).expanduser()
    path = project / "reports" / name
    return path if path.is_dir() else path.with_suffix(".zip")


def read_tables(path, names):
    if path.is_dir():
        return {name: pd.read_csv(path / f"{name}.csv") for name in names}
    if not path.is_file():
        raise FileNotFoundError(f"Missing prerequisite report: {path}")
    with ZipFile(path) as bundle:
        tables = {}
        for name in names:
            files = [file for file in bundle.namelist() if PurePosixPath(file).name == f"{name}.csv"]
            if len(files) != 1:
                raise ValueError(f"Expected one {name}.csv in {path}.")
            with bundle.open(files[0]) as stream:
                tables[name] = pd.read_csv(stream)
        return tables


def require_unique(table, keys, label):
    if table[keys].isna().any().any() or table.duplicated(keys).any():
        raise ValueError(f"Missing or duplicate {keys} in {label}.")


source14 = report_path(NOTEBOOK_14_EXPORT, "lifeguard_mit_bit_switch_replay_results")
source16 = report_path(NOTEBOOK_16_EXPORT, "lifeguard_mit_bit_controller_benchmark_results")
source17 = report_path(NOTEBOOK_17_EXPORT, "lifeguard_mit_bit_multiblock_simulator_results")
output = (Path(OUTPUT_ROOT).expanduser() if OUTPUT_ROOT else project / "reports") / OUTPUT_NAME


torch.set_num_threads(1)
pilot = report_path(NOTEBOOK_18_EXPORT, PILOT_NAME)
raw_source = Path(DATASET_PATH).expanduser() if DATASET_PATH is not None else project / "Dataset" / "MIT-BIT"
if DATASET_PATH is None and not raw_source.is_dir():
    raw_source = raw_source.with_suffix(".zip")
NOMINAL_AH = 2.4
SEEDS = (42, 73, 101)
EVAL_DRAWS = 8
TRAIN_DRAWS = 2
SAC_UPDATES = 1500
SAC_BATCH = 64
TAU = .005
TAIL_FRACTION = .10
FORECAST_TARGET = np.array([.20, .15, .10])
POLICIES = {"fixed": "Fixed 1C", "original": "Notebook 18 MPC", "improved": "Forecast-refined MPC", "sac": "Discrete SAC"}
BASE = "ridge"
CANDIDATES = {
    "ridge": "Notebook 18 Ridge",
    "linear": "Anchored linear correction",
    "kernel_20": "Kernel correction with 20% cycle features",
    "kernel_45": "Kernel correction with 45% cycle features",
    "kernel_voltage": "Kernel correction with voltage-based Wh",
    "kernel_raw": "Kernel correction with completed raw traces",
    "boost": "Shallow boosting correction",
    "gru": "Compact GRU residual",
}
FIELDS_OUT = ("wh", "minutes")

In [2]:
@dataclass(frozen=True)
class Block:
    start: int
    rate: int
    ah: float
    wh: float
    first_ah: float
    last_ah: float
    minutes: float


def build_trace(block_audit, cycle_audit, timing):
    require_unique(block_audit, KEYS, "Notebook 14 blocks")
    require_unique(cycle_audit, ["cell_id", "aging_cycle"], "Notebook 14 cycles")
    require_unique(timing, ["cell_id", "aging_cycle"], "Notebook 16 cycle times")
    time_blocks = timing.groupby(KEYS).charge_minutes.sum(min_count=BLOCK)
    raw = {int(cell): group.set_index("aging_cycle").sort_index()
           for cell, group in cycle_audit.groupby("cell_id")}
    traces = defaultdict(dict)
    for row in block_audit.itertuples(index=False):
        if row.status != "ok" or row.charge_rate_c not in RATES:
            continue
        start = int(row.start_cycle)
        part = raw[row.cell_id].reindex(range(start, start + BLOCK))
        capacities = part.capacity_ah.to_numpy(dtype=float)
        if np.isfinite(capacities).all():
            minutes = time_blocks.get((row.cell_id, start), np.nan)
            traces[row.cell_id][start] = Block(start, int(row.charge_rate_c), float(np.median(capacities)),
                                              float(row.observed_wh), capacities[0], capacities[-1], float(minutes))
    return dict(traces)


def initial_history(trace, start, require_energy=True):
    if start - BLOCK not in trace or start - 2 * BLOCK not in trace:
        return None
    history = tuple(trace[cycle] for cycle in sorted(trace) if cycle < start)
    if require_energy and (not np.isfinite(history[-1].wh) or history[-1].wh <= 0):
        return None
    return history


def feature_row(history):
    previous, latest = history[-2:]
    start = latest.start + BLOCK
    result = {
        "start_ah": latest.ah, "latest_ah": latest.last_ah,
        "recent_block_delta_ah": latest.ah - previous.ah,
        "within_block_delta_ah": latest.last_ah - latest.first_ah,
        "age_cycles": start - 1, "first_rate_c": history[0].rate,
        "previous_rate_c": latest.rate,
        "prior_2c_blocks": sum(block.rate == 2 for block in history),
        "prior_3c_blocks": sum(block.rate == 3 for block in history),
    }
    for rate in RATES:
        same = [block for block in history if block.rate == rate][-3:]
        last = same[-1].ah if same else np.nan
        gap = start - same[-1].start if same else np.nan
        slope = (last - same[0].ah) / (same[-1].start - same[0].start) if len(same) > 1 else 0
        result[f"last_{rate}c_ah"] = last
        result[f"lag_{rate}c_cycles"] = gap
        result[f"projected_{rate}c_ah"] = min(last, last + gap * slope) if same else np.nan
    return result


def prepare_examples(events, traces):
    origins, features, labels = [], [], []
    for row in events.itertuples(index=False):
        trace = traces[row.cell_id]
        history = initial_history(trace, row.start_cycle, require_energy=False)
        if history is None:
            continue
        block = trace.get(row.start_cycle)
        origins.append((int(row.cell_id), int(row.start_cycle), int(row.fold), int(row.charge_rate_c)))
        features.append(feature_row(history))
        if block is None:
            labels.append((np.nan,) * 5 + (history[-1].wh,))
            continue
        if row.charge_rate_c != block.rate:
            raise ValueError(f"Recorded rate disagreement for cell {row.cell_id}, cycle {row.start_cycle}.")
        np.testing.assert_allclose(row.observed_wh, block.wh, atol=1e-9, rtol=0)
        labels.append((block.ah, block.wh, block.first_ah - block.ah,
                       block.last_ah - block.ah, block.minutes, history[-1].wh))
    return (pd.DataFrame(origins, columns=[*KEYS, "fold", "charge_rate_c"]),
            pd.DataFrame(features),
            pd.DataFrame(labels, columns=["ah", "wh", "first_offset_ah", "last_offset_ah", "minutes", "previous_wh"]))


def recorded_windows(traces, origins, cohort):
    windows = []
    for cell, start, model_fold, rate in origins[[*KEYS, "fold", "charge_rate_c"]].itertuples(index=False, name=None):
        if not FIRST_START <= start <= LAST_START:
            continue
        trace = traces[cell]
        history = initial_history(trace, start)
        if history is None:
            continue
        recorded = []
        for step in range(max(HORIZONS)):
            cycle = start + step * BLOCK
            block = trace.get(cycle)
            if cycle > LAST_START or block is None or not np.isfinite([block.ah, block.wh, block.minutes]).all():
                break
            recorded.append(block)
        if recorded:
            if recorded[0].rate != rate:
                raise ValueError("Recorded window rate disagrees with the audited assignment.")
            windows.append(dict(cohort=cohort, cell_id=cell, start_cycle=start, fold=model_fold,
                                history=history, path=tuple(recorded)))
    return windows


def validate_folds(events, splits):
    require_unique(events, KEYS, "Notebook 14 predictions")
    require_unique(splits, ["fold", "cell_id"], "Notebook 14 cell splits")
    if set(splits.fold) != set(range(1, 6)) or events.cell_id.nunique() != 55:
        raise ValueError("Notebook 15 expects the full 55-cell, five-fold Notebook 14 benchmark.")
    test_fold = splits.loc[splits.role.eq("test")].set_index("cell_id", verify_integrity=True).fold
    cells = set(events.cell_id)
    if set(test_fold.index) != cells or not events.cell_id.map(test_fold).eq(events.fold).all():
        raise ValueError("Notebook 14 prediction rows and held-out folds disagree.")
    expected = {(fold, cell, "test" if test_fold[cell] == fold else "train")
                for fold in range(1, 6) for cell in cells}
    if set(splits[["fold", "cell_id", "role"]].itertuples(index=False, name=None)) != expected:
        raise ValueError("The saved train/test split is incomplete.")


audit14 = read_tables(source14, ("predictions", "cell_splits", "block_audit", "cycle_audit", "source_audit"))
audit16 = read_tables(source16, ("charge_time_audit", "calibration"))
audit17 = read_tables(source17, ("policy_trajectories", "recorded_rollout_totals"))
events = audit14["predictions"].sort_values(KEYS).reset_index(drop=True)
validate_folds(events, audit14["cell_splits"])
traces = build_trace(audit14["block_audit"], audit14["cycle_audit"], audit16["charge_time_audit"])
examples, features, targets = prepare_examples(events, traces)
windows = recorded_windows(traces, examples, "randomized")
complete_paths = [row for row in windows if len(row["path"]) == BLOCKS]
reference_paths = audit17["recorded_rollout_totals"]
reference_paths = reference_paths.loc[reference_paths.cohort.eq("randomized")
                                      & reference_paths.horizon_cycles.eq(BLOCK * BLOCKS)]
if ({(row["cell_id"], row["start_cycle"]) for row in complete_paths}
        != set(reference_paths[KEYS].itertuples(index=False, name=None))):
    raise ValueError("The complete recorded windows do not reproduce Notebook 17.")

In [3]:
SHEET = "xl/worksheets/sheet1.xml"
HEADERS = ("Data_Point", "Test_Time(s)", "Current(A)", "Capacity(Ah)", "Voltage(V)",
           "Energy(Wh)", "Temperature(℃)", "Date_Time", "Cycle_Index")
FIELDS = b"CDEFHI"
CELL_VALUE = re.compile(rb'<c r="([A-I])(\d+)"[^>]*><v>([^<]*)</v></c>')
NUMERIC = re.compile(rb"[+\-0-9.Ee]+")
STRING = re.compile(rb"<si(?:\s[^>]*)?>.*?</si>", re.DOTALL)



def validate_header(workbook, row):
    entries = CELL_VALUE.findall(row)
    if len(entries) != len(HEADERS):
        raise ValueError("Unexpected MIT–BIT worksheet header layout.")
    table = workbook.read("xl/sharedStrings.xml")
    wanted = {int(value) for _, _, value in entries}
    labels = {}
    for index, match in enumerate(STRING.finditer(table)):
        if index in wanted:
            labels[index] = "".join(ElementTree.fromstring(match[0]).itertext())
            if len(labels) == len(wanted):
                break
    if tuple(labels.get(int(value)) for _, _, value in entries) != HEADERS:
        raise ValueError("Unexpected MIT–BIT worksheet columns or units.")


def trace_chunks(workbook):
    # Stream the verified worksheet instead of building a workbook object with millions of cells.
    with workbook.open(SHEET) as xml:
        pending = xml.read(2**20)
        header_end = pending.find(b"</row>")
        if header_end < 0:
            raise ValueError("Missing worksheet header.")
        validate_header(workbook, pending[:header_end])
        pending = pending[header_end + len(b"</row>"):]
        while pending:
            cutoff = pending.rfind(b"</row>")
            if cutoff < 0:
                if b"</sheetData>" in pending:
                    break
                chunk = xml.read(2**20)
                if not chunk:
                    raise ValueError("Incomplete worksheet XML.")
                pending += chunk
                continue
            cutoff += len(b"</row>")
            complete, pending = pending[:cutoff], pending[cutoff:]
            values = [(col, value) for col, _, value in CELL_VALUE.findall(complete)
                      if col[0] in FIELDS]
            rows = complete.count(b"</row>")
            if (len(values) != rows * len(FIELDS)
                    or any(col[0] != FIELDS[i % len(FIELDS)] or not NUMERIC.fullmatch(value)
                           for i, (col, value) in enumerate(values))):
                raise ValueError("Incomplete numeric fields or reordered columns in the aging worksheet.")
            sample = np.fromstring(b" ".join(value for _, value in values), sep=" ").reshape(-1, len(FIELDS))
            yield sample
            pending += xml.read(2**20)


def cycle_traces(workbook):
    carry = np.empty((0, len(FIELDS)))
    previous = 0
    for chunk in trace_chunks(workbook):
        data = np.concatenate((carry, chunk))
        split = np.flatnonzero(data[:-1, -1] != data[1:, -1]) + 1
        groups = np.split(data, split)
        for trace in groups[:-1]:
            raw = trace[0, -1]
            if raw <= previous or raw != int(raw):
                raise ValueError("Repeated or unordered raw cycle index.")
            previous = int(raw)
            yield previous, trace
        carry = groups[-1]
    if carry.size:
        raw = carry[0, -1]
        if raw <= previous or raw != int(raw):
            raise ValueError("Repeated or unordered final cycle index.")
        yield int(raw), carry



@contextmanager
def open_aging_workbook(path, archive, name, zip_paths):
    if archive is None:
        with ZipFile(path / name) as workbook:
            yield workbook
        return
    with archive.open(zip_paths[name]) as compressed, SpooledTemporaryFile(max_size=4 * 2**20) as stream:
        while chunk := compressed.read(2**20):
            stream.write(chunk)
        stream.seek(0)
        with ZipFile(stream) as workbook:
            yield workbook


TEST_TIME_CELL = re.compile(rb'<c r="B(\d+)"([^>]*)><v>([^<]+)</v></c>')
ELAPSED_TIME = re.compile(r"(?:(\d+)-)?(\d{1,2}):(\d{2}):(\d{2})")


def test_time_seconds(workbook, first_row, last_row):
    entries = {}
    with workbook.open(SHEET) as xml:
        pending = b""
        while chunk := xml.read(2**20):
            pending += chunk
            cutoff = pending.rfind(b"</row>")
            if cutoff < 0:
                continue
            cutoff += len(b"</row>")
            for match in TEST_TIME_CELL.finditer(pending[:cutoff]):
                row = int(match[1])
                if first_row <= row <= last_row:
                    entries[row] = (b't="s"' in match[2], match[3])
                if row >= last_row:
                    break
            if last_row in entries:
                break
            pending = pending[cutoff:]
    if sorted(entries) != list(range(first_row, last_row + 1)):
        return np.full(last_row - first_row + 1, np.nan)

    wanted = {int(value) for is_string, value in entries.values() if is_string}
    labels = {}
    if wanted:
        for index, match in enumerate(STRING.finditer(workbook.read("xl/sharedStrings.xml"))):
            if index in wanted:
                labels[index] = "".join(ElementTree.fromstring(match[0]).itertext())
                if len(labels) == len(wanted):
                    break

    seconds = []
    for row in range(first_row, last_row + 1):
        is_string, value = entries[row]
        if not is_string:
            # The workbook formats numeric elapsed times as fractions of an Excel day.
            seconds.append(float(value) * 86400)
            continue
        clock = ELAPSED_TIME.fullmatch(labels.get(int(value), ""))
        if clock is None:
            return np.full(last_row - first_row + 1, np.nan)
        days, hour, minute, second = (int(part or 0) for part in clock.groups())
        seconds.append(days * 86400 + hour * 3600 + minute * 60 + second)
    return np.asarray(seconds)


def valid_clock(seconds):
    return (np.isfinite(seconds).all() and np.all(np.diff(seconds) >= -1.1)
            and np.all(np.diff(np.maximum.accumulate(seconds)) <= 30))


def charge_duration(trace, expected_rate, workbook, first_sheet_row):
    current, _, _, _, date_days, _ = trace.T
    if not np.isfinite(current).all() or not np.isfinite(date_days).all():
        return np.nan, np.nan, "invalid_trace"
    negative = np.flatnonzero(current < -0.1)
    stop = negative[0] if negative.size else len(current)
    positive = np.flatnonzero(current[:stop] > 0.1)
    if positive.size < 5 or positive[-1] == len(current) - 1 or np.any(np.diff(positive) != 1):
        return np.nan, np.nan, "incomplete_or_interrupted_charge"
    samples = current[positive]
    amps = NOMINAL_AH * expected_rate
    if (abs(np.quantile(samples, 0.9) - amps) > 0.05 * amps
            or np.sum(abs(samples - amps) <= 0.05 * amps) < 5):
        raise ValueError(f"Raw workbook does not verify the audited {expected_rate}C charge.")
    wall_clock = (date_days[positive] - date_days[positive[0]]) * 86400
    wall_minutes = float(wall_clock[-1] / 60)
    seconds, status = wall_clock, "ok_wall_clock"
    if not valid_clock(wall_clock):
        start, end = first_sheet_row + positive[[0, -1]]
        elapsed = test_time_seconds(workbook, start, end)
        if not valid_clock(elapsed):
            return np.nan, wall_minutes, "invalid_timestamps"
        seconds, status = elapsed - elapsed[0], "ok_test_time"
    duration = float(np.maximum.accumulate(seconds)[-1] / 60)
    return (duration, wall_minutes, status) if duration > 0 else (np.nan, wall_minutes, "invalid_timestamps")




def charge_shape(trace, workbook, row):
    i, _, voltage, _, date, _ = trace.T
    discharge = np.flatnonzero(i < -.1)
    if len(discharge) < 5:
        return None
    positive = np.flatnonzero(i[:discharge[0]] > .1)
    if len(positive) < 5 or np.any(np.diff(positive) != 1):
        return None
    seconds = (date[positive] - date[positive[0]]) * 86400
    if not valid_clock(seconds):
        test = test_time_seconds(workbook, row + positive[0], row + positive[-1])
        if not valid_clock(test):
            return None
        seconds = test - test[0]
    elapsed = np.maximum.accumulate(seconds)
    values = voltage[positive]
    ramp = []
    for threshold in (3.8, 4.0, 4.1):
        indices = np.flatnonzero(values >= threshold)
        ramp.append(float(elapsed[indices[0]] / 60) if len(indices) else np.nan)
    charge_ah = np.dot(np.diff(elapsed), i[positive[1:]] + i[positive[:-1]]) / 7200
    discharge_elapsed = np.maximum.accumulate((date[discharge] - date[discharge[0]]) * 86400)
    at_15 = min(np.searchsorted(discharge_elapsed, 15), len(discharge)-1)
    at_60 = min(np.searchsorted(discharge_elapsed, 60), len(discharge)-1)
    drop = voltage[discharge[0]] - voltage[discharge[at_60]]
    return [*ramp, charge_ah, elapsed[-1] / 60, values[0], voltage[discharge[at_15]], drop]


def extract_shapes(path, manifest):
    fields = ["to_38_min", "to_40_min", "to_41_min", "charge_ah", "charge_min", "start_v", "discharge_v_15", "discharge_drop_60"]
    chosen = manifest.loc[manifest.status.eq("read"), ["cell_id", "workbook"]]
    require_unique(chosen, ["cell_id"], "MIT–BIT workbook manifest")
    if len(chosen) != 55:
        raise ValueError("The raw trace experiment expects all 55 randomized batteries.")
    records = []
    with ZipFile(path) if path.is_file() else _empty_archive() as archive:
        relevant = {}
        if archive is not None:
            required = set(chosen.workbook.to_list())
            for member in archive.namelist():
                suffix = "/".join(PurePosixPath(member).parts[-2:])
                if suffix in required:
                    if suffix in relevant:
                        raise ValueError(f"Duplicate MIT–BIT workbook {suffix}.")
                    relevant[suffix] = member
        for number, (cell, filename) in enumerate(chosen.itertuples(index=False, name=None), 1):
            with open_aging_workbook(path, archive, filename, relevant) as workbook:
                row = 2
                for cycle, trace in cycle_traces(workbook):
                    first_row = row
                    row += len(trace)
                    if cycle > LAST_START + BLOCK:
                        break
                    if cycle > 1:
                        shape = charge_shape(trace, workbook, first_row)
                        if shape is not None:
                            records.append((int(cell), cycle - 1, *shape))
            if number % 10 == 0 or number == len(chosen):
                print(f"Read raw charge histories for {number}/{len(chosen)} batteries.", flush=True)
    return pd.DataFrame(records, columns=["cell_id", "aging_cycle", *fields])


@contextmanager
def _empty_archive():
    yield None


with threadpool_limits(limits=1):
    raw_shapes = extract_shapes(raw_source, audit14["source_audit"])
print(f"Verified raw features for {len(raw_shapes):,} charging cycles.")

Verified raw features for 5,225 charging cycles.


## Forecast features and training

Each candidate receives only completed history and the planned rate prefix. The 3C discharge-only boundary supplies one standardized pre-aging measurement for each battery. Cycle features summarize observed capacity and energy in the latest block and in the most recent interval at each rate. The raw-trace variant also receives the preceding 5-cycle voltage crossing times, charge Ah, duration and initial or load voltage. Simulated intermediate steps keep the last measured trace and use simulated scalar block outcomes.

The existing one-step Ridge supplies a candidate trajectory. Error-correction models improve cumulative totals separately at 5, 10, and 15 cycles. Wh is corrected in Wh. Charging time is corrected on the log scale from a capacity-based anchor. Kernel distances use separate normalized groups for the basic history, individual cycles and raw waveform features.

Each outer validation battery fold is untouched by selection. Inner validation residual models receive cross-fitted training Ridge trajectories from other folds. Per outcome and horizon, select the model with the lowest weighted relative mean, tail and maximum error among those that improve or match all three metrics on inner validation. Ridge is the fallback. Inner GRU selection uses one seed and the outer GRU averages two.

In [4]:
def design_matrices(rows):
    next_rate = rows.charge_rate_c.to_numpy()
    previous = rows.previous_rate_c.to_numpy()
    rate_flags = np.column_stack([(next_rate == rate) for rate in (2, 3)]
                                 + [(previous == rate) for rate in (2, 3)])
    transition_flags = np.column_stack([(previous == a) & (next_rate == b)
                                        for a in RATES for b in RATES if (a, b) != (1, 1)])
    start = rows.start_ah.to_numpy()
    history = rows[["recent_block_delta_ah", "within_block_delta_ah", "age_cycles",
                    "prior_3c_blocks", "prior_2c_blocks"]].to_numpy()
    common = np.column_stack((start, rows.latest_ah, rate_flags, transition_flags, history))
    interactions = start[:, None] * rate_flags
    same = rows[["last_same_rate_ah", "projected_same_rate_ah", "lag_same_rate_cycles"]].to_numpy()
    per_rate = rows[[*(f"last_{rate}c_ah" for rate in RATES),
                     *(f"projected_{rate}c_ah" for rate in RATES),
                     *(f"lag_{rate}c_cycles" for rate in RATES)]].to_numpy()
    return {
        "standard": np.column_stack((common, interactions, history[:, :2] * rate_flags[:, :2])),
        "expanded": np.column_stack((common, same, per_rate, interactions)),
    }


def action_features(history, previous_rate, candidate):
    table = history.copy()
    table["previous_rate_c"] = previous_rate
    table["charge_rate_c"] = candidate
    for feature, suffix in (("last", "ah"), ("projected", "ah"), ("lag", "cycles")):
        history_by_rate = table[[f"{feature}_{rate}c_{suffix}" for rate in RATES]].to_numpy()
        rate_index = np.broadcast_to(np.asarray(candidate, dtype=int), len(table)) - 1
        table[f"{feature}_same_rate_{suffix}"] = history_by_rate[np.arange(len(table)), rate_index]
    return design_matrices(table)


def fit_ridge(matrix, labels, cells):
    sample = pd.Series(cells)
    weights = 1 / sample.groupby(sample).transform("size").to_numpy(dtype=float)
    weights /= weights.mean()
    usable = np.isfinite(matrix).any(axis=0)
    observed = matrix[:, usable]
    missing = ~np.isfinite(observed)
    indicators = missing.any(axis=0)
    medians = np.nanmedian(np.where(missing, np.nan, observed), axis=0)

    def fill(data):
        absent = ~np.isfinite(data)
        return np.column_stack((np.where(absent, medians, data), absent[:, indicators]))

    filled = fill(observed)
    center = np.average(filled, weights=weights, axis=0)
    scale = np.maximum(np.sqrt(np.average((filled - center)**2, weights=weights, axis=0)), 1e-8)
    fit = Ridge(alpha=1).fit((filled - center) / scale, labels, sample_weight=weights)

    def predict(rows):
        return fit.predict((fill(rows[:, usable]) - center) / scale)
    return predict


def medians_by_rate(train_examples, train_targets):
    values = train_targets.assign(rate=train_examples.charge_rate_c.to_numpy())
    medians = values.groupby("rate")[["ah", "wh", "minutes"]].median().reindex(RATES)
    if not np.isfinite(medians.to_numpy()).all():
        raise ValueError("The training cohort lacks per-rate baseline measurements.")
    return medians.to_numpy()


def fit_simulator(origin, features, targets, include):
    design = action_features(features, features.previous_rate_c, origin.charge_rate_c)
    battery = origin.cell_id.to_numpy()
    specs = {
        "ah": ("standard", "expanded"), "wh": ("standard", "expanded"),
        "first_offset_ah": ("standard",), "last_offset_ah": ("standard",), "minutes": ("standard",),
    }
    training = {}
    for target, families in specs.items():
        values = targets[target].to_numpy()
        usable = include & np.isfinite(values)
        if target == "wh":
            usable &= targets.previous_wh.notna().to_numpy()
        if target == "minutes":
            values = np.where(values > 0, values, np.nan)
            usable &= np.isfinite(values)
            values = np.log(values)
        if not usable.any():
            raise ValueError(f"No training data for {target}.")
        training[target] = {family: fit_ridge(design[family][usable], values[usable], battery[usable])
                            for family in families}

    def predict(rows, candidate):
        features = action_features(rows, rows.previous_rate_c, candidate)
        values = {}
        for target, models in training.items():
            estimate = models["standard"](features["standard"])
            if "expanded" in models:
                estimate = 0.75 * estimate + 0.25 * models["expanded"](features["expanded"])
            values[target] = estimate
        values["ah"] = np.maximum(values["ah"], 0)
        values["wh"] = np.maximum(values["wh"], 0)
        values["minutes"] = np.exp(values["minutes"])
        values["first_ah"] = np.maximum(values["ah"] + values.pop("first_offset_ah"), 0)
        values["last_ah"] = np.maximum(values["ah"] + values.pop("last_offset_ah"), 0)
        return pd.DataFrame(values, index=rows.index)

    return predict, medians_by_rate(origin.loc[include], targets.loc[include])


def run_rollouts(scenarios, simulator, medians, policy=None, calibration=None):
    if not scenarios:
        return pd.DataFrame()
    histories = [item["history"] for item in scenarios]
    steps_limit = max(HORIZONS) if policy else max(len(item["path"]) for item in scenarios)
    fixed_rate = {"fixed_1c": 1, "fixed_2c": 2, "fixed_3c": 3}.get(policy)
    outcomes = []
    for step in range(steps_limit):
        active = [i for i, item in enumerate(scenarios) if policy or step < len(item["path"])]
        states = [histories[i] for i in active]
        context = pd.DataFrame(feature_row(history) for history in states)
        target = ENERGY_TARGET * np.asarray([history[-1].wh for history in states])
        fallback = np.zeros(len(states), dtype=bool)
        if policy is None:
            rates = np.asarray([scenarios[i]["path"][step].rate for i in active])
        elif fixed_rate is not None:
            rates = np.full(len(states), fixed_rate)
        else:
            speeds = calibration.training_typical_charge_minutes.to_numpy()
            if policy == "rule":
                scores = np.asarray([[latest_measured(history, rate, "wh") for rate in RATES]
                                     for history in states])
            else:
                forecast = np.column_stack([simulator(context, rate).wh for rate in RATES])
                scores = np.maximum(0, forecast - calibration.overprediction_margin_wh.to_numpy())
            rates, fallback = choose_rate(scores, speeds, target)
        predictions = simulator(context, rates)
        if not np.isfinite(predictions.to_numpy()).all():
            raise ValueError("The simulator returned a nonfinite state.")
        for j, (i, rate, values) in enumerate(zip(active, rates, predictions.itertuples(index=False))):
            scenario = scenarios[i]
            initial = scenario["history"]
            start = scenario["start_cycle"] + step * BLOCK
            batch = Block(start, int(rate), values.ah, values.wh, values.first_ah, values.last_ah, values.minutes)
            histories[i] += (batch,)
            record = dict(cohort=scenario["cohort"], cell_id=scenario["cell_id"], fold=scenario["fold"],
                          start_cycle=scenario["start_cycle"], block_start=start, step=step + 1, rate=int(rate),
                          predicted_ah=values.ah, predicted_wh=values.wh, predicted_minutes=values.minutes,
                          initial_wh=initial[-1].wh, moving_target_wh=target[j], fallback=bool(fallback[j]))
            if policy is None:
                observed = scenario["path"][step]
                baselines = baseline_values(initial, rate, medians)
                record.update(observed_ah=observed.ah, observed_wh=observed.wh, observed_minutes=observed.minutes,
                              baseline_ah=baselines[0], baseline_wh=baselines[1], baseline_minutes=baselines[2])
            else:
                record["policy"] = policy
            outcomes.append(record)
    return pd.DataFrame(outcomes)


def latest_measured(history, rate, field, fallback=np.nan):
    return next((getattr(block, field) for block in reversed(history)
                 if block.rate == rate and np.isfinite(getattr(block, field))), fallback)


def baseline_values(history, rate, medians):
    return [latest_measured(history, rate, field, median)
            for (field, median) in zip(("ah", "wh", "minutes"), medians[rate - 1])]


def initial_policy_states(traces, examples):
    parts = []
    for cell, group in examples.groupby("cell_id"):
        fold = int(group.fold.iloc[0])
        for start in POLICY_STARTS:
            history = initial_history(traces[cell], start)
            if history is not None:
                parts.append(dict(cohort="randomized", cell_id=cell, start_cycle=start, fold=fold, history=history))
    return parts


def forecast_paths(histories, sequences, simulator):
    sequences = np.asarray(sequences, dtype=int)
    roots, plans = len(histories), len(sequences)
    # Evaluate each distinct prefix once per root and carry the next state to its children.
    states = {(): histories}
    wh = {(): np.zeros(roots)}
    minutes = {(): np.zeros(roots)}
    first_blocks = {}
    for depth in range(1, sequences.shape[1] + 1):
        prefixes = sorted({tuple(row[:depth]) for row in sequences})
        history_rows = [history for prefix in prefixes for history in states[prefix[:-1]]]
        rate_rows = np.repeat([prefix[-1] for prefix in prefixes], roots)
        context = pd.DataFrame(feature_row(history) for history in history_rows)
        values = simulator(context, rate_rows)
        if not np.isfinite(values.to_numpy()).all():
            raise ValueError("The simulator returned a nonfinite schedule forecast.")
        aligned = iter(values.itertuples(index=False))
        next_states, next_wh, next_minutes = {}, {}, {}
        for prefix in prefixes:
            prior = states[prefix[:-1]]
            blocks = [Block(history[-1].start + BLOCK, prefix[-1], row.ah, row.wh,
                            row.first_ah, row.last_ah, row.minutes)
                      for history, row in zip(prior, islice(aligned, roots), strict=True)]
            next_states[prefix] = [history + (block,) for history, block in zip(prior, blocks)]
            next_wh[prefix] = wh[prefix[:-1]] + np.array([block.wh for block in blocks])
            next_minutes[prefix] = minutes[prefix[:-1]] + np.array([block.minutes for block in blocks])
            if depth == 1:
                first_blocks[prefix[-1]] = blocks
        states, wh, minutes = next_states, next_wh, next_minutes
    assert len(wh) == plans
    return (np.column_stack([wh[tuple(row)] for row in sequences]),
            np.column_stack([minutes[tuple(row)] for row in sequences]), first_blocks)


def budget_for_start(histories, simulator):
    fixed_path = [(1,) * BLOCKS]
    fixed_wh, _, _ = forecast_paths(histories, fixed_path, simulator)
    return fixed_wh[:, 0], ENERGY_TARGET * fixed_wh[:, 0]


def historical_contexts():
    cycles = audit14["cycle_audit"].set_index(["cell_id", "aging_cycle"], verify_integrity=True)
    cells = {int(cell): part.droplevel(0)[["capacity_ah", "delivered_wh"]]
             for cell, part in cycles.groupby(level=0)}
    raw = {int(cell): part.set_index("aging_cycle").drop(columns="cell_id")
           for cell, part in raw_shapes.groupby("cell_id")}
    return cells, raw


def safe_mean(values):
    return float(np.mean(values[np.isfinite(values)])) if np.isfinite(values).any() else np.nan


CYCLE_HISTORY, RAW_HISTORY = historical_contexts()


def measured_snapshot(cell, start, history):
    cycle = CYCLE_HISTORY[cell]
    window = cycle.reindex(range(start - 10, start)).to_numpy()
    initial_ah, initial_wh = cycle.loc[0].to_numpy()
    capacity, energy = window.T
    voltage = energy / np.maximum(capacity, .01)
    last = history[-1]
    cyc = [
        initial_ah, initial_wh, initial_wh / initial_ah, last.ah / initial_ah, last.wh / (5 * initial_wh),
        capacity[-1], energy[-1], voltage[-1], capacity[-1] - capacity[-5], energy[-1] - energy[-5],
        voltage[-1] - voltage[-5], np.std(capacity[-5:]), np.std(energy[-5:]),
        safe_mean(capacity[-5:]) - safe_mean(capacity[:5]), safe_mean(energy[-5:]) - safe_mean(energy[:5]),
        *capacity[-5:], *energy[-5:],
    ]
    effective = []
    for rate in RATES:
        same = [block for block in history if block.rate == rate]
        period = cycle.reindex(range(same[-1].start, same[-1].start + BLOCK)).to_numpy() if same else None
        if period is None:
            cyc.extend([np.nan] * 6)
            eff = np.nan
        else:
            a, b = period.T
            eff = np.sum(b) / np.sum(a) if np.isfinite(period).all() and np.sum(a) > 0 else np.nan
            cyc.extend((a[-1] - a[0], b[-1] - b[0], np.std(a), a[-1] / initial_ah, safe_mean(b) / initial_wh, eff))
        effective.append(float(eff) if np.isfinite(eff) else safe_mean(voltage))
    raw = RAW_HISTORY[cell]
    raw_recent = raw.reindex(range(start - 5, start)).mean().to_numpy()
    raw_older = raw.reindex(range(start - 10, start - 5)).mean().to_numpy()
    raw_initial = raw.reindex(range(1, 6)).mean().to_numpy()
    return tuple(cyc), (*raw_recent, *(raw_recent - raw_older), *raw_initial), tuple(effective)


def state_history(history, start):
    last, previous = history[-2:][::-1]
    core = [
        last.ah, last.wh, last.minutes, last.rate, last.ah - previous.ah,
        last.wh - previous.wh, start, len(history),
    ]
    for rate in RATES:
        similar = [b for b in history if b.rate == rate]
        for field in ("ah", "wh", "minutes"):
            available = [(b.start, getattr(b, field)) for b in similar if np.isfinite(getattr(b, field))]
            if not available:
                core.extend((np.nan, np.nan, np.nan))
            else:
                cycle, value = available[-1]
                slope = ((value - available[-2][1]) / (cycle - available[-2][0])) if len(available) > 1 else np.nan
                core.extend((value, start - cycle, slope))
    for idx in range(6):
        if idx < len(history):
            b = history[-1-idx]
            core.extend((b.rate, b.ah, b.wh, b.minutes, b.last_ah - b.first_ah))
        else:
            core.extend((np.nan,) * 5)
    return core


def tensors(history):
    result = np.zeros((6, 11), dtype=np.float32)
    for j, b in enumerate(history[-6:], start=max(0, 6-len(history))):
        result[j] = (1, b.start/100, b.rate==1, b.rate==2, b.rate==3, b.ah/2.4, b.last_ah/2.4,
                     5*(b.last_ah-b.first_ah), b.wh/40 if np.isfinite(b.wh) else 0,
                     b.minutes/250 if np.isfinite(b.minutes) else 0, float(np.isfinite(b.minutes)))
    return result


def schedule_features(core, snapshots, rates, paths):
    # paths contains predicted [Ah, Wh, minutes] for each prescribed action.
    ah, wh, minutes = np.moveaxis(paths, 2, 0)
    physics = 125 * np.maximum(ah, .005) / rates
    counts = np.column_stack([(rates == rate).sum(axis=1) for rate in RATES])
    planned = np.eye(4)[rates][:, :, 1:]
    extra, raw, voltage = (np.asarray(part) for part in zip(*snapshots))
    totals = np.column_stack((wh.sum(axis=1), minutes.sum(axis=1), physics.sum(axis=1)))
    voltage_anchor = (5 * ah * voltage[np.arange(len(rates))[:, None], rates-1]).sum(axis=1)
    voltage_anchor = np.where(np.isfinite(voltage_anchor), voltage_anchor, totals[:, 0])
    matrix = np.column_stack((core[:, :8], counts, planned.reshape(len(rates), -1), core[:, 8:],
                             ah.mean(axis=1), ah[:, -1]-ah[:, 0], totals, ah.std(axis=1), ah, wh, physics, minutes))
    return matrix, extra, raw, totals, voltage_anchor


BASE_MODELS, ROLLOUTS = {}, {}


def fit_base(train_folds):
    key = tuple(sorted(train_folds))
    if key not in BASE_MODELS:
        BASE_MODELS[key] = fit_simulator(examples, features, targets, examples.fold.isin(key).to_numpy())
    return BASE_MODELS[key]


def recorded_roll(training, test):
    key = tuple(sorted(training)), test
    if key not in ROLLOUTS:
        predictor, medians = fit_base(training)
        held = [case for case in windows if case["fold"] == test]
        trace = run_rollouts(held, predictor, medians)
        ROLLOUTS[key] = trace.sort_values([*KEYS, "step"])
    return ROLLOUTS[key]


SNAPSHOTS = {(case["cell_id"], case["start_cycle"]):
             measured_snapshot(case["cell_id"], case["start_cycle"], case["history"]) for case in windows}
HISTORY = {(case["cell_id"], case["start_cycle"]): case["history"] for case in windows}


def measured_pack(data, horizon):
    keys, rates, history, snapshots, reference, realized = [], [], [], [], [], []
    for key, group in data.groupby(KEYS, sort=True):
        if len(group) < horizon:
            continue
        part = group.head(horizon)
        keys.append(key)
        rates.append(part.rate.to_numpy())
        history.append(HISTORY[key])
        snapshots.append(SNAPSHOTS[key])
        reference.append(part[["predicted_ah", "predicted_wh", "predicted_minutes"]].to_numpy())
        realized.append(part[["observed_wh", "observed_minutes"]].to_numpy())
    inputs = schedule_features(np.asarray([state_history(state, key[1]) for key, state in zip(keys, history)]),
                               snapshots, np.stack(rates), np.stack(reference))
    return dict(keys=keys, inputs=inputs, rates=np.stack(rates), paths=np.stack(reference),
                measured=np.stack(realized), history=history)

In [5]:
def pack_join(parts):
    numeric = ("rates", "paths", "measured")
    combined = {name: np.concatenate([part[name] for part in parts]) for name in numeric}
    combined["inputs"] = tuple(np.concatenate(items) for items in zip(*(part["inputs"] for part in parts)))
    combined["keys"] = [key for part in parts for key in part["keys"]]
    combined["history"] = [state for part in parts for state in part["history"]]
    return combined


PACKS = {}


def window_pack(training, scored, horizon):
    key = tuple(sorted(training)), scored, horizon
    if key not in PACKS:
        PACKS[key] = measured_pack(recorded_roll(training, scored), horizon)
    return PACKS[key]


def cross_fitted(pool, horizon):
    return pack_join([window_pack(pool - {validation}, validation, horizon) for validation in sorted(pool)])


def cell_weights(keys):
    cells = pd.Series([key[0] for key in keys])
    weights = 1 / cells.groupby(cells).transform("size").to_numpy(dtype=float)
    return weights / weights.mean()


def standardize(training, weights):
    usable = np.isfinite(training).any(axis=0)
    matrix = training[:, usable]
    missing = ~np.isfinite(matrix)
    indicators = missing.any(axis=0)
    medians = np.nanmedian(np.where(missing, np.nan, matrix), axis=0)

    def fill(matrix):
        x = matrix[:, usable]
        missing = ~np.isfinite(x)
        return np.column_stack((np.where(missing, medians, x), missing[:, indicators]))

    x = fill(training)
    mean = np.average(x, axis=0, weights=weights)
    scale = np.maximum(np.sqrt(np.average((x - mean)**2, axis=0, weights=weights)), 1e-8)
    return lambda matrix: np.clip((fill(matrix)-mean)/scale, -6, 6)


def fitted_heads(pack):
    core, cycle, raw, typical, voltage = pack["inputs"]
    actual = pack["measured"].sum(axis=1)
    weights = cell_weights(pack["keys"])
    center = [standardize(values, weights) for values in (core, cycle, raw)]
    matrices = [transform(features) for transform, features in zip(center, (core, cycle, raw))]
    distances = [euclidean_distances(values, squared=True) / values.shape[1] for values in matrices]
    fit = {}
    physics_time_error = np.log(actual[:, 1] / typical[:, 2])
    ordinary = np.column_stack((actual[:, 0] - typical[:, 0], physics_time_error))

    linear = Ridge(alpha=25).fit(matrices[0], ordinary, sample_weight=weights)

    def linear_forecast(inputs):
        pred = linear.predict(center[0](inputs[0]))
        return np.column_stack((inputs[3][:, 0] + pred[:, 0], inputs[3][:, 2] * np.exp(pred[:, 1])))
    fit["linear"] = linear_forecast

    kernel_choices = {
        "kernel_20": ((.8, .2, 0), False),
        "kernel_45": ((.55, .45, 0), False),
        "kernel_voltage": ((.8, .2, 0), True),
        "kernel_raw": ((.65, .2, .15), False),
    }
    for name, (contributions, use_voltage) in kernel_choices.items():
        distance = sum(c * distances[j] for j, c in enumerate(contributions) if c)
        gram = np.exp(-distance)
        gram[np.diag_indices_from(gram)] += .25 / weights
        reward = ordinary if not use_voltage else np.column_stack((actual[:, 0]-voltage, physics_time_error))
        dual = solve(gram, reward, assume_a="pos", check_finite=False)

        def forecast(inputs, contributions=contributions, dual=dual, use_voltage=use_voltage):
            distance = sum(c * euclidean_distances(center[j](inputs[j]), matrices[j], squared=True) / matrices[j].shape[1]
                           for j, c in enumerate(contributions) if c)
            correction = np.exp(-distance) @ dual
            wh = inputs[4] if use_voltage else inputs[3][:, 0]
            return np.column_stack((wh + correction[:, 0], inputs[3][:, 2] * np.exp(correction[:, 1])))
        fit[name] = forecast

    training = np.column_stack((core, cycle, raw))
    boosting = []
    for target in ordinary.T:
        model = HistGradientBoostingRegressor(max_depth=3, max_iter=110, min_samples_leaf=23,
            l2_regularization=3, learning_rate=.035, early_stopping=False, random_state=SEED)
        model.fit(training, target, sample_weight=weights)
        boosting.append(model)

    def boost_forecast(inputs):
        matrix = np.column_stack(inputs[:3])
        return np.column_stack((inputs[3][:, 0] + boosting[0].predict(matrix),
                                inputs[3][:, 2] * np.exp(boosting[1].predict(matrix))))
    fit["boost"] = boost_forecast
    return fit


class ResidualGRU(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder = nn.GRU(11, 16, batch_first=True)
        self.decoder = nn.GRU(6, 16, batch_first=True)
        self.head = nn.Sequential(nn.Linear(16, 12), nn.SiLU(), nn.Linear(12, 2))

    def forward(self, history, planned):
        _, state = self.encoder(history)
        trace, _ = self.decoder(planned, state)
        return self.head(trace)


def planned_tokens(rates, paths):
    ah, wh, time = np.moveaxis(paths, 2, 0)
    known = np.stack((wh/35, (np.log(time)-4)/2, ah/2.4), axis=2)
    return np.concatenate((np.eye(4)[rates][:, :, 1:], known), axis=2).astype(np.float32)


def fit_gru(pack, seeds):
    H = torch.tensor(np.stack([tensors(history) for history in pack["history"]]))
    P = torch.tensor(planned_tokens(pack["rates"], pack["paths"]))
    delta_wh = pack["measured"][:, :, 0] - pack["paths"][:, :, 1]
    delta_minutes = np.log(pack["measured"][:, :, 1] / pack["paths"][:, :, 2])
    change = torch.tensor(np.stack((delta_wh/.65, delta_minutes/.045), axis=2), dtype=torch.float32)
    weights = torch.tensor(cell_weights(pack["keys"])[:, None, None], dtype=torch.float32)
    models = []
    for seed in seeds:
        torch.manual_seed(seed)
        model = ResidualGRU()
        optimizer = torch.optim.AdamW(model.parameters(), lr=.003, weight_decay=.02)
        for _ in range(70):
            optimizer.zero_grad()
            prediction = model(H, P)
            per_block = nn.functional.smooth_l1_loss(prediction, change, reduction="none")
            cumulative = (prediction - change).cumsum(dim=1)
            loss = (weights * (per_block + .3 * nn.functional.smooth_l1_loss(
                cumulative, torch.zeros_like(cumulative), reduction="none"))).mean()
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1)
            optimizer.step()
        models.append(model.eval())
    return models


def gru_forecast(models, history, rates, paths):
    H = torch.tensor(np.stack([tensors(state) for state in history]))
    inputs = torch.tensor(planned_tokens(rates, paths))
    with torch.no_grad():
        correction = torch.stack([model(H, inputs) for model in models]).mean(dim=0).numpy()
    return np.column_stack(((paths[:, :, 1] + .65*correction[:, :, 0]).sum(axis=1),
                            (paths[:, :, 2] * np.exp(.045*correction[:, :, 1])).sum(axis=1)))


def score_arrays(error, cells):
    abs_error = np.abs(error)
    n_tail = int(np.ceil(len(abs_error) * TAIL_FRACTION))
    return np.array((pd.Series(abs_error).groupby(np.asarray(cells)).mean().mean(),
                     np.sort(abs_error)[-n_tail:].mean(), abs_error.max()))


FITTED_CATALOGS = {}


def train_forecaster(pool, seeds):
    key = tuple(sorted(pool)), seeds
    if key not in FITTED_CATALOGS:
        packs = {h: cross_fitted(pool, h) for h in HORIZONS}
        heads = {h: fitted_heads(pack) for h, pack in packs.items()}
        gru = fit_gru(packs[BLOCKS], seeds)
        FITTED_CATALOGS[key] = heads, gru
    return FITTED_CATALOGS[key]


def predict_candidates(bundle, pack, horizon, names=None):
    heads, gru = bundle
    names = tuple(CANDIDATES) if names is None else names
    result = {}
    for name in names:
        if name == "ridge":
            value = pack["inputs"][3][:, :2]
        elif name == "gru":
            value = gru_forecast(gru, pack["history"], pack["rates"], pack["paths"])
        else:
            value = heads[horizon][name](pack["inputs"])
        result[name] = np.maximum(value, .001)
    return result


def inner_selection(pool, outer):
    inner_errors = defaultdict(list)
    for held in sorted(pool):
        train = pool - {held}
        bundle = train_forecaster(train, SEEDS[:1])
        for h in HORIZONS:
            test = window_pack(train, held, h)
            actual = test["measured"].sum(axis=1)
            cells = [k[0] for k in test["keys"]]
            for name, predicted in predict_candidates(bundle, test, h).items():
                inner_errors[h, name].append((predicted - actual, cells))

    selected, records, margins = {}, [], {}
    ratios = np.array([.5, .3, .2])
    for h in HORIZONS:
        have = {}
        for name in CANDIDATES:
            parts = inner_errors[h, name]
            errors = np.concatenate([values for values, _ in parts])
            cells = [cell for _, block in parts for cell in block]
            have[name] = np.stack([score_arrays(errors[:, target], cells) for target in range(len(FIELDS_OUT))])
        for col, field in enumerate(FIELDS_OUT):
            baseline = have["ridge"][col]
            qualifies = {model: bool(np.all(stats[col] <= baseline + 1e-9)) for model, stats in have.items()}
            cost = {model: float(np.dot(ratios, stats[col] / baseline)) for model, stats in have.items()}
            chosen = min((model for model in have if qualifies[model]), key=lambda name: (cost[name], name != "ridge"))
            selected[h, field] = chosen
            if field == "wh":
                residuals = np.concatenate([block[:, 0] for block, _ in inner_errors[h, chosen]])
                margins[h] = float(max(0, np.quantile(residuals, OVERPREDICTION_QUANTILE)))
            for model in have:
                fields = have[model][col]
                records.append((outer, h, field, model, *fields, cost[model], qualifies[model], model == chosen))
    return selected, records, margins


def evaluate_forecasts():
    rows, choices, models = [], [], {}
    for outer in FOLDS:
        pool = set(FOLDS) - {outer}
        selected, scores, margins = inner_selection(pool, outer)
        choices.extend(scores)
        bundle = train_forecaster(pool, SEEDS[:2])
        models[outer] = (bundle, selected, margins)
        for h in HORIZONS:
            test = window_pack(pool, outer, h)
            truth = test["measured"].sum(axis=1)
            values = predict_candidates(bundle, test, h)
            values["selected"] = np.column_stack([values[selected[h, target]][:, col] for col, target in enumerate(FIELDS_OUT)])
            for name, available in values.items():
                for key, pred, y, sequence in zip(test["keys"], available, truth, test["rates"]):
                    rows.append((*key, outer, h*BLOCK, name, "".join(map(str, sequence)), *y, *pred))
        print(f"Completed nested Wh and time selection for fold {outer}: {selected}", flush=True)
    return (pd.DataFrame(rows, columns=[*KEYS, "fold", "horizon_cycles", "model", "sequence",
                                      "observed_wh", "observed_minutes", "predicted_wh", "predicted_minutes"]),
            pd.DataFrame(choices, columns=["fold", "horizon_blocks", "target", "model", "inner_mae",
                                          "inner_tail", "inner_max", "inner_score", "eligible", "selected"]),
            models)


with threadpool_limits(limits=1):
    predictions, selections, validated_models = evaluate_forecasts()

Completed nested Wh and time selection for fold 1: {(1, 'wh'): 'kernel_raw', (1, 'minutes'): 'kernel_raw', (2, 'wh'): 'kernel_raw', (2, 'minutes'): 'kernel_raw', (3, 'wh'): 'kernel_voltage', (3, 'minutes'): 'kernel_raw'}
Completed nested Wh and time selection for fold 2: {(1, 'wh'): 'kernel_raw', (1, 'minutes'): 'kernel_raw', (2, 'wh'): 'kernel_raw', (2, 'minutes'): 'kernel_raw', (3, 'wh'): 'kernel_raw', (3, 'minutes'): 'kernel_raw'}
Completed nested Wh and time selection for fold 3: {(1, 'wh'): 'boost', (1, 'minutes'): 'kernel_raw', (2, 'wh'): 'ridge', (2, 'minutes'): 'kernel_raw', (3, 'wh'): 'kernel_voltage', (3, 'minutes'): 'kernel_raw'}
Completed nested Wh and time selection for fold 4: {(1, 'wh'): 'kernel_raw', (1, 'minutes'): 'kernel_raw', (2, 'wh'): 'kernel_raw', (2, 'minutes'): 'kernel_raw', (3, 'wh'): 'kernel_voltage', (3, 'minutes'): 'kernel_raw'}
Completed nested Wh and time selection for fold 5: {(1, 'wh'): 'kernel_voltage', (1, 'minutes'): 'kernel_raw', (2, 'wh'): 'kernel_

## Fair comparison of the charging policies

Keep each origin's Notebook 18 fixed-1C reference and 90% energy budget. The original MPC uses its original model and margins. Refined MPC and SAC share the same forecast guard. A rate is available when at least one supported completion can reach the remaining Wh budget after the training-calibrated margin. A fully infeasible state falls back to 1C and is counted.

All policies are scored in the same frozen Ridge transition simulator. It uses joint Wh, time and Ah disturbances from sequences of one-step training-fold errors. The new cumulative forecast is used to plan rather than to score its own actions. Test disturbances and training draws use separate streams. The original MPC's deterministic baseline is verified separately against Notebook 18. Measured exact-match replay provides an additional check.

SAC trains from simulation replay of all legal branches of training batteries. The policy and both Q networks use two 64-unit layers. It optimizes masked categorical expectations with twin target critics. Reward is negative charge minutes divided by 100. The terminal reward also subtracts ten times the fraction of the energy budget missed. The entropy target is 70% of the log of the legal action count. The agent is evaluated by maximum legal action probability. The three test seeds share the same scenarios.

In [6]:
def plan_bank(histories, sequences, simulator):
    roots = len(histories)
    states, paths = {(): histories}, {(): np.empty((roots, 0, 3))}
    first = {}
    for depth in range(1, sequences.shape[1] + 1):
        prefixes = sorted({tuple(row[:depth]) for row in sequences})
        parents = [history for p in prefixes for history in states[p[:-1]]]
        rates = np.repeat([p[-1] for p in prefixes], roots)
        estimates = simulator(pd.DataFrame(feature_row(history) for history in parents), rates)
        values = estimates[["ah", "wh", "minutes"]].to_numpy().reshape(len(prefixes), roots, 3)
        blocks = estimates[["ah", "wh", "first_ah", "last_ah", "minutes"]].to_numpy().reshape(len(prefixes), roots, 5)
        next_states, next_paths = {}, {}
        for index, prefix in enumerate(prefixes):
            previous = states[prefix[:-1]]
            current = [Block(history[-1].start + BLOCK, int(prefix[-1]), *row)
                       for history, row in zip(previous, blocks[index])]
            next_states[prefix] = [history + (block,) for history, block in zip(previous, current)]
            next_paths[prefix] = np.concatenate((paths[prefix[:-1]], values[index, :, None]), axis=1)
            if depth == 1:
                first[int(prefix[0])] = current
        states, paths = next_states, next_paths
    return np.stack([paths[tuple(row)] for row in sequences], axis=1), first


def ranking(wh, minutes, budget, margin, sequences):
    conservative = np.maximum(0, wh - margin)
    feasible = conservative + 1e-9 >= budget[:, None]
    choices = np.empty(len(wh), dtype=int)
    fallback = ~feasible.any(axis=1)
    for i, accepted in enumerate(feasible):
        if fallback[i]:
            possible = np.flatnonzero(sequences[:, 0] == 1)
            choices[i] = possible[np.lexsort((possible, minutes[i, possible], -wh[i, possible]))[0]]
        else:
            possible = np.flatnonzero(accepted)
            choices[i] = possible[np.lexsort((possible, -wh[i, possible], minutes[i, possible]))[0]]
    legal = np.column_stack([feasible[:, sequences[:, 0] == rate].any(axis=1) for rate in RATES])
    legal[fallback, 0] = True
    return sequences[choices, 0], legal, fallback


def interval_plans(histories, snapshots, credit, budget, remaining, simulator, validated, old_margin, original=True):
    sequences = np.asarray(list(product(RATES, repeat=remaining)))
    paths, first = plan_bank(histories, sequences, simulator)
    base_wh, base_min = paths[..., 1].sum(axis=2), paths[..., 2].sum(axis=2)
    original_choice, _, original_fallback = ranking(base_wh, base_min, budget-credit, old_margin, sequences)
    if original:
        return original_choice, original_fallback, first

    bundle, selection, margins = validated
    branches = len(sequences)
    core = np.asarray([state_history(history, history[-1].start + BLOCK) for history in histories])
    flattened = paths.reshape(-1, remaining, 3)
    rates = np.tile(sequences, (len(histories), 1))
    contexts = [state for state in histories for _ in sequences]
    inputs = schedule_features(np.repeat(core, branches, axis=0),
                              [snapshot for snapshot in snapshots for _ in sequences], rates, flattened)
    names = {selection[remaining, field] for field in FIELDS_OUT}
    heads = predict_candidates(bundle, dict(inputs=inputs, paths=flattened, rates=rates, history=contexts),
                               remaining, names)
    wh, minutes = (heads[selection[remaining, field]][:, index].reshape(len(histories), branches)
                   for index, field in enumerate(FIELDS_OUT))
    choice, legal, fallback = ranking(wh, minutes, budget-credit, margins[remaining], sequences)
    best_times = np.column_stack([minutes[:, sequences[:, 0] == rate].min(axis=1) for rate in RATES])
    best_energies = np.column_stack([wh[:, sequences[:, 0] == rate].max(axis=1) for rate in RATES])
    latest = [story[-1] for story in histories]
    observation = np.column_stack((
        np.asarray([block.ah/2.4 for block in latest]),
        np.asarray([block.wh/40 for block in latest]),
        np.asarray([block.minutes/250 if np.isfinite(block.minutes) else 0 for block in latest]),
        np.asarray([np.isfinite(block.minutes) for block in latest]),
        np.asarray([(block.start+BLOCK-1)/100 for block in latest]),
        np.eye(4)[[block.rate for block in latest]][:, 1:],
        (budget-credit)/np.maximum(budget, 1),
        budget/100, np.full(len(histories), remaining/BLOCKS),
        best_times/600, best_energies/100,
    )).astype(np.float32)
    return dict(original=original_choice, original_fallback=original_fallback, improved=choice, legal=legal,
                fallback=fallback, observation=observation, first=first)


def noise_paths(outer):
    pool = set(FOLDS) - {outer}
    residuals = {}
    for inner in sorted(pool):
        table = recorded_roll(pool - {inner}, inner)
        for row in table.loc[table.step.eq(1)].itertuples(index=False):
            residuals[row.cell_id, row.start_cycle] = np.asarray((
                row.observed_ah-row.predicted_ah, row.observed_wh-row.predicted_wh,
                np.log(row.observed_minutes/row.predicted_minutes),
            ))
    records = [np.stack([residuals[case["cell_id"], case["start_cycle"]+BLOCK*offset] for offset in range(BLOCKS)])
               for case in complete_paths if case["fold"] != outer
               and all((case["cell_id"], case["start_cycle"]+BLOCK*offset) in residuals for offset in range(BLOCKS))]
    if not records:
        raise ValueError("No independent training residual paths.")
    return np.stack(records)


def advance_block(block, noise):
    ah = max(.005, block.ah + float(noise[0]))
    shift = ah - block.ah
    return Block(block.start, block.rate, ah, max(.001, block.wh+float(noise[1])),
                 max(.005, block.first_ah+shift), max(.005, block.last_ah+shift),
                 block.minutes*np.exp(float(noise[2])))


def old_margins(outer):
    table = read_tables(pilot, ("calibration_margins",))["calibration_margins"]
    return table.loc[table.fold.eq(outer)].set_index("remaining_blocks").margin_wh


def build_tree(cases, noise, outer, restrict=False):
    # Noise is applied only after an action. Nodes never expose a future draw to a policy.
    histories = [case["history"] for case in cases]
    snapshots = [measured_snapshot(case["cell_id"], case["start_cycle"], case["history"]) for case in cases]
    simulator, _ = fit_base(set(FOLDS) - {outer})
    margin = old_margins(outer)
    fixed, budget = budget_for_start(histories, simulator)
    trees = [dict() for _ in cases]
    active = [(i, (), history, 0., 0.) for i, history in enumerate(histories)]
    for turn in range(BLOCKS):
        children = []
        for start in range(0, len(active), 96):
            batch = active[start:start+96]
            idx = np.asarray([row[0] for row in batch], dtype=int)
            state = [row[2] for row in batch]
            credited = np.asarray([row[3] for row in batch])
            guesses = interval_plans(state, [snapshots[i] for i in idx], credited, budget[idx],
                                     BLOCKS-turn, simulator, validated_models[outer], margin[BLOCKS-turn],
                                     original=False)
            for j, (i, path, history, wh, minutes) in enumerate(batch):
                node = {key: guesses[key][j] for key in
                        ("original", "original_fallback", "improved", "legal", "fallback", "observation")}
                node.update(wh=wh, minutes=minutes, budget=budget[i], fixed=fixed[i])
                trees[i][path] = node
                allowed = np.asarray(RATES)[node["legal"]] if restrict else RATES
                for rate in allowed:
                    block = advance_block(guesses["first"][rate][j], noise[i][turn])
                    children.append((i, path + (rate,), history + (block,), wh+block.wh, minutes+block.minutes))
        active = children
    for i, path, _, wh, minutes in active:
        trees[i][path] = dict(wh=wh, minutes=minutes, budget=budget[i], fixed=fixed[i])
    return trees


def enumerate_samples(trees):
    data = []
    for tree in trees:
        for path, current in tree.items():
            if len(path) == BLOCKS:
                continue
            for option in np.flatnonzero(current["legal"]):
                after = tree[path + (int(option)+1,)]
                done = len(path) + 1 == BLOCKS
                reward = -(after["minutes"]-current["minutes"])/100
                if done:
                    reward -= 10 * max(0, current["budget"] - after["wh"])/max(current["budget"], 1)
                data.append((current["observation"], current["legal"], option, reward,
                             after["observation"] if not done else np.zeros_like(current["observation"]),
                             after["legal"] if not done else np.array([True, False, False]),
                             done))
    return [np.asarray(items) for items in zip(*data)]


def run_tree(tree, policy, actor=None):
    path = ()
    fallbacks = 0
    while len(path) < BLOCKS:
        node = tree[path]
        if policy == "fixed":
            rate = 1
        elif policy == "original":
            rate = int(node["original"])
            fallbacks += node["original_fallback"]
        elif policy == "improved":
            rate = int(node["improved"])
            fallbacks += node["fallback"]
        else:
            rate = 1 + actor.choose(node["observation"], node["legal"])
            fallbacks += node["fallback"]
        if policy in ("improved", "sac"):
            assert node["legal"][rate - 1]
        path = path + (rate,)
    return dict(**tree[path], sequence="".join(map(str, path)), fallbacks=int(fallbacks))

In [7]:
def actor_or_critic(width):
    return nn.Sequential(nn.Linear(width, 64), nn.ReLU(), nn.Linear(64, 64), nn.ReLU(), nn.Linear(64, 3))


def legal_distribution(logits, mask):
    assert mask.any(dim=-1).all()
    log = torch.log_softmax(logits.masked_fill(~mask, -1e9), dim=-1)
    probs = torch.where(mask, log.exp(), 0)
    return probs, torch.where(mask, log, 0)


class DiscreteSAC:
    def __init__(self, width):
        self.actor = actor_or_critic(width)
        self.q1, self.q2 = actor_or_critic(width), actor_or_critic(width)
        self.target1, self.target2 = actor_or_critic(width), actor_or_critic(width)
        self.target1.load_state_dict(self.q1.state_dict())
        self.target2.load_state_dict(self.q2.state_dict())
        self.log_alpha = nn.Parameter(torch.tensor(-2.3025851))

    def choose(self, observation, mask):
        return int(self.choose_batch(np.atleast_2d(observation), np.atleast_2d(mask))[0])

    def choose_batch(self, obs, mask):
        with torch.no_grad():
            logits = self.actor(torch.as_tensor(obs, dtype=torch.float32))
            return logits.masked_fill(~torch.as_tensor(mask), -1e9).argmax(dim=1).numpy()

    def fit(self, raw, seed):
        optimizer = torch.optim.Adam(self.actor.parameters(), lr=.0003)
        critic = torch.optim.Adam(list(self.q1.parameters())+list(self.q2.parameters()), lr=.0003)
        entropy_optimizer = torch.optim.Adam([self.log_alpha], lr=.0003)
        observed, masks, action, rewards, next_obs, next_masks, terminal = [
            torch.as_tensor(value, dtype=dtype) for value, dtype in
            zip(raw, (torch.float32, torch.bool, torch.long, torch.float32,
                      torch.float32, torch.bool, torch.float32))
        ]
        rng = np.random.default_rng(seed)
        records = []
        for step in range(1, SAC_UPDATES + 1):
            ix = rng.integers(len(observed), size=SAC_BATCH)
            x, valid, a, r, y, future, done = (
                observed[ix], masks[ix], action[ix], rewards[ix], next_obs[ix], next_masks[ix], terminal[ix])
            with torch.no_grad():
                p, logp = legal_distribution(self.actor(y), future)
                future_min = torch.minimum(self.target1(y), self.target2(y))
                estimated = r + (1-done) * (p * (future_min-self.log_alpha.exp()*logp)).sum(dim=1)
            q1 = self.q1(x).gather(1, a[:, None]).squeeze(1)
            q2 = self.q2(x).gather(1, a[:, None]).squeeze(1)
            qloss = nn.functional.smooth_l1_loss(q1, estimated) + nn.functional.smooth_l1_loss(q2, estimated)
            critic.zero_grad()
            qloss.backward()
            critic.step()

            p, logp = legal_distribution(self.actor(x), valid)
            with torch.no_grad():
                minimum = torch.minimum(self.q1(x), self.q2(x))
            aloss = (p * (self.log_alpha.detach().exp()*logp-minimum)).sum(dim=1).mean()
            optimizer.zero_grad()
            aloss.backward()
            optimizer.step()

            entropy = -(p * logp).sum(dim=1).detach()
            alpha_loss = (self.log_alpha * (entropy - .7*valid.sum(dim=1).float().log())).mean()
            entropy_optimizer.zero_grad()
            alpha_loss.backward()
            entropy_optimizer.step()
            with torch.no_grad():
                self.log_alpha.clamp_(-7, 0)
                for source, target in ((self.q1, self.target1), (self.q2, self.target2)):
                    for a, b in zip(source.parameters(), target.parameters()):
                        b.lerp_(a, TAU)
            if step % 250 == 0:
                if not torch.isfinite(qloss + aloss):
                    raise ValueError("SAC training produced a nonfinite loss.")
                records.append((step, float(qloss.detach()), float(aloss.detach()), float(self.log_alpha.exp().detach())))
        self.actor.eval()
        return records


def verify_distribution():
    p, log = legal_distribution(torch.ones((2, 3)), torch.tensor([[1, 0, 1], [0, 1, 0]], dtype=torch.bool))
    torch.testing.assert_close(p, torch.tensor([[.5, 0, .5], [0, 1, 0]]))
    assert torch.isfinite(log).all()


verify_distribution()

In [8]:
PILOT_SIMULATION = read_tables(pilot, ("simulation_totals",))["simulation_totals"]


def guard_reference(cases, outer):
    histories = [case["history"] for case in cases]
    reference, budget = budget_for_start(histories, fit_base(set(FOLDS) - {outer})[0])
    keys = [(case["cell_id"], case["start_cycle"]) for case in cases]
    expected = PILOT_SIMULATION.loc[PILOT_SIMULATION.policy.eq("fixed_1c")].set_index(KEYS).loc[keys]
    np.testing.assert_allclose(reference, expected.wh, rtol=0, atol=1e-9)
    np.testing.assert_allclose(budget, expected.budget_wh, rtol=0, atol=1e-9)

    simulator, _ = fit_base(set(FOLDS)-{outer})
    credit, minute = np.zeros(len(cases)), np.zeros(len(cases))
    sequence = [""] * len(cases)
    for remaining in reversed(HORIZONS):
        action, _, blocks = interval_plans(histories, None, credit, budget, remaining, simulator, None,
                                          old_margins(outer)[remaining], original=True)
        for i, rate in enumerate(action):
            block = blocks[rate][i]
            histories[i] += (block,)
            credit[i] += block.wh
            minute[i] += block.minutes
            sequence[i] += str(rate)
    baseline = PILOT_SIMULATION.loc[PILOT_SIMULATION.policy.eq("energy_budget")].set_index(KEYS).loc[keys]
    np.testing.assert_allclose(credit, baseline.wh, rtol=0, atol=1e-9)
    np.testing.assert_allclose(minute, baseline.minutes, rtol=0, atol=1e-9)
    np.testing.assert_array_equal(sequence, baseline.sequence.astype(str))


def controller_trials():
    starts = initial_policy_states(traces, examples)
    if len(starts) != 164:
        raise ValueError("A benchmark start does not match Notebook 18.")
    rows, log, learned, replay_counts = [], [], {}, []
    actors = {}
    for outer in FOLDS:
        train = [s for s in starts if s["fold"] != outer]
        test = [s for s in starts if s["fold"] == outer]
        assert set(item["cell_id"] for item in train).isdisjoint(item["cell_id"] for item in test)
        guard_reference(test, outer)
        patterns = noise_paths(outer)
        tr = [case for case in train for _ in range(TRAIN_DRAWS)]
        rng = np.random.default_rng(np.random.SeedSequence([SEED, outer, 11]))
        replay = enumerate_samples(build_tree(tr, patterns[rng.integers(len(patterns), size=len(tr))], outer, restrict=True))
        assert replay[1][np.arange(len(replay[0])), replay[2]].all()
        for action in range(3):
            replay_counts.append((outer, action+1, int(np.sum(replay[2] == action))))
        actors[outer] = {}
        for seed in SEEDS:
            torch.manual_seed(seed)
            agent = DiscreteSAC(replay[0].shape[1])
            entries = agent.fit(replay, seed)
            log.extend((outer, seed, *row) for row in entries)
            actors[outer][seed] = agent
            learned[f"fold-{outer}/seed-{seed}"] = agent.actor.state_dict()

        te = [case for case in test for _ in range(EVAL_DRAWS)]
        evaluate_rng = np.random.default_rng(np.random.SeedSequence([SEED, outer, 42]))
        trees = build_tree(te, patterns[evaluate_rng.integers(len(patterns), size=len(te))], outer)
        for i, (case, tree) in enumerate(zip(te, trees)):
            for policy in POLICIES:
                models = actors[outer].items() if policy == "sac" else [(0, None)]
                for seed, actor in models:
                    outcome = run_tree(tree, policy, actor)
                    rows.append(dict(cell_id=case["cell_id"], start_cycle=case["start_cycle"], fold=outer,
                                     draw=i % EVAL_DRAWS, policy=policy, seed=seed, **outcome))
        print(f"Fold {outer}: {len(test)} shared test starts, {len(replay[0]):,} legal SAC training transitions.", flush=True)
    return (pd.DataFrame(rows),
            pd.DataFrame(log, columns=["fold", "seed", "updates", "critic_loss", "actor_loss", "entropy_temperature"]),
            pd.DataFrame(replay_counts, columns=["fold", "rate", "training_transitions"]), actors, learned)


with threadpool_limits(limits=1):
    controller_paths, training_log, sac_replay_sizes, actors, weights = controller_trials()

Fold 1: 33 shared test starts, 1,688 legal SAC training transitions.
Fold 2: 33 shared test starts, 1,649 legal SAC training transitions.
Fold 3: 33 shared test starts, 1,649 legal SAC training transitions.
Fold 4: 32 shared test starts, 1,670 legal SAC training transitions.
Fold 5: 33 shared test starts, 1,691 legal SAC training transitions.


In [9]:
def replay_origins(examples, traces):
    logged = examples.set_index(KEYS, verify_integrity=True).charge_rate_c.to_dict()
    upper = LAST_START - BLOCK * (BLOCKS - 1)
    states = []
    for cell, start, fold in examples[[*KEYS, "fold"]].itertuples(index=False, name=None):
        if not FIRST_START <= start <= upper:
            continue
        history = initial_history(traces[cell], start)
        if history is None:
            continue
        cycles = [start + offset * BLOCK for offset in range(BLOCKS)]
        rates = tuple(logged.get((cell, cycle)) for cycle in cycles)
        if all(rate in RATES for rate in rates):
            states.append(dict(cell_id=cell, start_cycle=start, fold=fold, history=history, logged_rates=rates))
    return states


def replay_collected_paths():
    states = replay_origins(examples, traces)
    if len(states) != 819:
        raise ValueError("Measured-replay eligibility differs from Notebook 18.")
    result = []
    for outer in FOLDS:
        simulator, _ = fit_base(set(FOLDS)-{outer})
        supported = [row for row in states if row["fold"] == outer]
        _, budgets = budget_for_start([row["history"] for row in supported], simulator)
        margin = old_margins(outer)
        for policy in ("improved", "sac"):
            choices = actors[outer].items() if policy == "sac" else [(0, None)]
            for seed, actor in choices:
                paths = [{**case, "budget": float(budget), "wh": [], "time": [], "decisions": [],
                          "termination": "completed"}
                         for case, budget in zip(supported, budgets)]
                active = paths.copy()
                for turn in range(BLOCKS):
                    if not active:
                        break
                    history = [row["history"] for row in active]
                    snaps = [measured_snapshot(row["cell_id"], row["start_cycle"] + turn*BLOCK, row["history"])
                             for row in active]
                    remaining = BLOCKS - turn
                    estimates = interval_plans(history, snaps, np.asarray([sum(row["wh"]) for row in active]),
                        np.asarray([row["budget"] for row in active]), remaining, simulator,
                        validated_models[outer], margin[remaining], original=False)
                    possible = estimates["improved"] if policy == "improved" else 1 + actor.choose_batch(
                        estimates["observation"], estimates["legal"])
                    assert estimates["legal"][np.arange(len(active)), possible-1].all()
                    next_states = []
                    for case, action in zip(active, possible):
                        if action != case["logged_rates"][turn]:
                            case["termination"] = "action_mismatch"
                            continue
                        observed = traces[case["cell_id"]].get(case["start_cycle"]+BLOCK*turn)
                        case["decisions"].append(str(action))
                        case["wh"].append(observed.wh if observed is not None else np.nan)
                        case["time"].append(observed.minutes if observed is not None else np.nan)
                        if turn+1 < BLOCKS:
                            if observed is None or not np.isfinite(observed.wh):
                                case["termination"] = "missing_feedback"
                                continue
                            case["history"] += (observed,)
                        next_states.append(case)
                    active = next_states
                for case in paths:
                    complete = len(case["decisions"]) == BLOCKS
                    result.append(dict(cell_id=case["cell_id"], start_cycle=case["start_cycle"], fold=outer,
                        policy=policy, seed=seed, matched_steps=len(case["decisions"]), full_match=complete,
                        sequence="".join(case["decisions"]), termination=case["termination"], budget_wh=case["budget"],
                        measured_wh=np.sum(case["wh"]) if complete else np.nan,
                        measured_minutes=np.sum(case["time"]) if complete else np.nan))
    return pd.DataFrame(result)


def replay_summary(results):
    rows = []
    for (policy, seed), group in results.groupby(["policy", "seed"]):
        w = 27 / group.groupby("cell_id").start_cycle.transform("size")
        all_measured = group.full_match & group.measured_wh.notna() & group.measured_minutes.notna()
        wh = group.full_match & group.measured_wh.notna()
        time = group.full_match & group.measured_minutes.notna()
        rows.append(dict(policy=policy, seed=seed, eligible=len(group), full_matches=int(group.full_match.sum()),
                         complete=int(all_measured.sum()), batteries=group.loc[all_measured, "cell_id"].nunique(),
                         budget_hits=int((group.loc[wh, "measured_wh"] >= group.loc[wh, "budget_wh"]).sum()),
                         wh_matches=int(wh.sum()), time_matches=int(time.sum()),
                         measured_minutes=np.average(group.loc[time, "measured_minutes"], weights=w[time]) if time.any() else np.nan,
                         measured_wh=np.average(group.loc[wh, "measured_wh"], weights=w[wh]) if wh.any() else np.nan))
    archived = read_tables(pilot, ("replay_metrics",))["replay_metrics"].set_index("policy")
    for name, old in (("fixed", "fixed_1c"), ("original", "energy_budget")):
        row = archived.loc[old]
        rows.append(dict(policy=name, seed=0, eligible=int(row.origins), full_matches=int(row.full_matches),
                         complete=int(row.complete_outcomes), batteries=int(row.complete_batteries),
                         budget_hits=int(row.measured_budget_hits), wh_matches=int(row.energy_matches),
                         time_matches=int(row.time_matches), measured_minutes=row.measured_minutes,
                         measured_wh=row.measured_wh))
    return pd.DataFrame(rows)


with threadpool_limits(limits=1):
    measured_replay = replay_collected_paths()
    measured_metrics = replay_summary(measured_replay)
print(measured_metrics[["policy", "seed", "full_matches", "complete", "batteries"]].to_string(index=False))

  policy  seed  full_matches  complete  batteries
improved     0            28        28         23
     sac    42            26        26         22
     sac    73            26        26         22
     sac   101            26        26         22
   fixed     0            21        19         18
original     0            24        24         18


In [10]:
def bootstrap_delta(values):
    rng = np.random.default_rng(SEED)
    array = np.asarray(values)
    selected = rng.integers(len(array), size=(BOOTSTRAP_DRAWS, len(array)))
    return np.quantile(array[selected].mean(axis=1), [.025,.975])


def summarize_forecasts(predictions):
    metrics, cases = [], []
    for h, horizon in predictions.groupby("horizon_cycles"):
        ref = horizon.loc[horizon.model.eq("ridge")].set_index(KEYS)
        expected = {5:923, 10:862, 15:801}
        if h in (5, 15) and len(ref) != expected[h]:
            raise ValueError(f"Held-out support changed at horizon {h}.")
        for model, subset in horizon.groupby("model"):
            actual = subset.set_index(KEYS)
            pd.testing.assert_index_equal(actual.index.sort_values(), ref.index.sort_values())
            for field in FIELDS_OUT:
                target = "observed_" + field
                pred = "predicted_" + field
                e = (actual[pred]-actual[target]).abs()
                b = (ref[pred]-ref[target]).abs().reindex(actual.index)
                stats = score_arrays(e, e.index.get_level_values("cell_id"))
                pivot = score_arrays(b, b.index.get_level_values("cell_id"))
                worst = e.idxmax()
                pair = (e-b).groupby(level=0).mean()
                lo, hi = bootstrap_delta(pair)
                fraction = (pivot-stats)/pivot
                metrics.append((h, model, field, len(e), *stats, *fraction, lo, hi,
                                int(worst[0]), int(worst[1]), bool(np.all(fraction + 1e-9 >= FORECAST_TARGET))))
                special = (77, 56) if field == "wh" else (59, 16)
                if special in e.index and model in {"ridge", "selected"}:
                    cases.append((h, field, model, *special, e.loc[special]))
    return (pd.DataFrame(metrics, columns=["horizon_cycles", "model", "target", "windows", "mae", "tail_mae", "max_error",
                                           "mean_reduction", "tail_reduction", "max_reduction", "mae_ci95_low",
                                           "mae_ci95_high", "worst_cell", "worst_origin", "all_improvement_targets"]),
            pd.DataFrame(cases, columns=["horizon_cycles", "target", "model", *KEYS, "error"]))


def summarize_controllers(data):
    rows = data.assign(hit=data.wh.ge(data.budget-1e-9),
                       shortfall=(data.budget-data.wh).clip(lower=0))
    columns = ["minutes", "wh", "hit", "shortfall", "fallbacks"]
    battery = rows.groupby(["policy", "cell_id"])[columns].mean()
    metric = battery.groupby(level=0).mean().reindex(POLICIES)
    for name, group in rows.groupby("policy"):
        metric.loc[name, "max_shortfall_wh"] = group.shortfall.max()
        metric.loc[name, "tail_shortfall_wh"] = np.sort(group.shortfall)[-int(np.ceil(len(group)*.10)):].mean()
        metric.loc[name, "episodes"] = len(group)
    records = []
    for challenger, previous in (("improved", "original"), ("sac", "original"), ("sac", "improved")):
        diff = battery.loc[challenger] - battery.loc[previous]
        time = float(diff.minutes.mean())
        lo, hi = bootstrap_delta(diff.minutes)
        hit = float(diff.hit.mean())
        hlo, hhi = bootstrap_delta(diff.hit)
        highest = metric.loc[challenger, "max_shortfall_wh"] - metric.loc[previous, "max_shortfall_wh"]
        passes = hi < 0 and hit >= -.01 - 1e-9 and highest <= 1e-9
        records.append((challenger, previous, time, lo, hi, hit, hlo, hhi, highest, passes))
    comparisons = pd.DataFrame(records, columns=["policy", "baseline", "time_change_min", "time_ci95_low",
        "time_ci95_high", "hit_rate_change", "hit_ci95_low", "hit_ci95_high", "max_shortfall_change_wh", "qualifies"])
    check = comparisons.set_index(["policy", "baseline"]).qualifies
    best = "improved" if check.loc["improved", "original"] else "original"
    metric["selected"] = metric.index == ("sac" if check.loc["sac", best] else best)
    return metric, comparisons


forecast_metrics, worst_cases = summarize_forecasts(predictions)
controller_metrics, comparisons = summarize_controllers(controller_paths)
print(forecast_metrics.loc[forecast_metrics.model.isin(["ridge","selected"])].round(3).to_string(index=False))
print(controller_metrics.round(3).to_string())
print(comparisons.round(3).to_string(index=False))

 horizon_cycles    model  target  windows   mae  tail_mae  max_error  mean_reduction  tail_reduction  max_reduction  mae_ci95_low  mae_ci95_high  worst_cell  worst_origin  all_improvement_targets
              5    ridge      wh      923 0.470     1.501      3.451           0.000           0.000          0.000         0.000          0.000          77            11                    False
              5    ridge minutes      923 2.692    10.761     24.411           0.000           0.000          0.000         0.000          0.000          59            36                    False
              5 selected      wh      923 0.300     1.001      3.203           0.362           0.333          0.072        -0.202         -0.141          77            11                    False
              5 selected minutes      923 1.252     4.752     11.216           0.535           0.558          0.541        -1.702         -1.190          59            36                     True
             10    r

In [11]:
def report_text():
    focus = forecast_metrics.loc[forecast_metrics.horizon_cycles.eq(15) & forecast_metrics.model.eq("selected")]
    winner = controller_metrics.index[controller_metrics.selected][0]
    lines = [
        "# MIT–BIT charging forecast refinement and Discrete SAC", "",
        "## Results", "",
        f"**Selected controller: {POLICIES[winner]}.** Forecasts were validated against measured charge timing and "
        "meter-checked delivered Wh. Controllers used a common Ridge transition simulator with training-fold "
        "disturbances. Each policy retained the original 90% energy budget.", "",
        "## Held-out forecast comparison", "",
        "| Horizon | Model | Outcome | Windows | Mean absolute error | Worst 10% mean | Maximum error |",
        "| ---: | --- | --- | ---: | ---: | ---: | ---: |",
    ]
    for row in forecast_metrics.loc[forecast_metrics.model.isin(["ridge", "selected"])].itertuples(index=False):
        label = "Selected per training fold" if row.model == "selected" else CANDIDATES[row.model]
        target = "Delivered Wh" if row.target == "wh" else "Charging minutes"
        lines.append(f"| {row.horizon_cycles} cycles | {label} | {target} | {row.windows} | "
                     f"{row.mae:.3f} | {row.tail_mae:.3f} | {row.max_error:.3f} |")
    lines.extend(["", "The final catalog includes Ridge, capacity-anchored linear correction, Kernel Ridge residual "
                  "correction, gradient boosting and a compact GRU. All candidate scores and fold selections are in the ZIP. "
                  "The same MIT–BIT cohort was used in model screening.", ""])
    for row in focus.itertuples(index=False):
        target = "Delivered energy" if row.target == "wh" else "Charging time"
        join = "met" if row.all_improvement_targets else "fell short of"
        lines.extend([
            f"{target} reduced mean, worst-decile and maximum errors by "
            f"{row.mean_reduction:.1%}, {row.tail_reduction:.1%} and {row.max_reduction:.1%}, respectively, "
            f"and {join} the full 20% / 15% / 10% target. Its mean error change had a paired 95% interval of "
            f"{row.mae_ci95_low:+.3f} to {row.mae_ci95_high:+.3f}.", "",
        ])
    lines.extend(["### Previously identified cases", "",
                  "| Battery | Start cycle | Target | Ridge absolute error at 15 cycles | Selected absolute error at 15 cycles |",
                  "| --- | ---: | --- | ---: | ---: |"])
    checks = worst_cases.loc[worst_cases.horizon_cycles.eq(15)]
    for (field, cell, start), group in checks.groupby(["target", *KEYS]):
        errors = group.set_index("model").error
        lines.append(f"| #{cell} | {start} | {field} | {errors['ridge']:.3f} | {errors['selected']:.3f} |")
    lines.extend([
        "", "## Controller simulation on 164 matched starts", "",
        "The original deterministic MPC and fixed 1C budgets reproduce Notebook 18. The table below includes eight "
        "matched disturbance paths per start. SAC is averaged across three training seeds. Means weight batteries equally.", "",
        "| Policy | Charging minutes | Delivered Wh | Budget hit rate | Worst Wh shortfall |",
        "| --- | ---: | ---: | ---: | ---: |",
    ])
    for name, row in controller_metrics.iterrows():
        lines.append(f"| {POLICIES[name]} | {row.minutes:.2f} | {row.wh:.2f} | {row.hit:.2%} | "
                     f"{row.max_shortfall_wh:.3f} |")
    lines.extend([
        "", "## Paired controller comparisons", "",
        "| Policy vs baseline | Charging-time difference (95% interval, min) | Budget hit change | "
        "Max Wh shortfall change | Qualifies |",
        "| --- | ---: | ---: | ---: | --- |",
    ])
    for row in comparisons.itertuples(index=False):
        lines.append(f"| {POLICIES[row.policy]} vs {POLICIES[row.baseline]} | "
                     f"{row.time_change_min:+.2f} ({row.time_ci95_low:+.2f} to {row.time_ci95_high:+.2f}) | "
                     f"{100*row.hit_rate_change:+.2f} pp | {row.max_shortfall_change_wh:+.3f} | "
                     f"{'Yes' if row.qualifies else 'No'} |")
    lines.extend([
        "",
        "Advancement requires a negative upper 95% confidence bound for charging time, a budget hit-rate decrease "
        "of at most one percentage point, and no increase in maximum Wh shortfall. Intervals resample batteries. "
        "The simulated outcomes cover 15 cycles with 3C discharge.", "",
        "## Recorded randomized replay", "",
        "| Policy | Seed | Full matches | Complete matches (batteries) | Measured minutes | Measured Wh |",
        "| --- | ---: | ---: | ---: | ---: | ---: |",
    ])
    for row in measured_metrics.itertuples(index=False):
        lines.append(f"| {POLICIES[row.policy]} | {row.seed or '—'} | {row.full_matches}/{row.eligible} | "
                     f"{row.complete} ({row.batteries}) | {row.measured_minutes:.2f} | {row.measured_wh:.2f} |")
    lines.extend([
        "", "Only actions that match all three recorded choices contribute to the outcome-specific replay estimates. "
        "Each policy can match a different set of trajectories. Replay means use normalized inverse propensity weights "
        "and equal battery weight before matching. The replay evidence is descriptive.", "",
        "[MIT–BIT v3 dataset](https://data.mendeley.com/datasets/kw34hhw7xg/3) · "
        "[Discrete SAC](https://arxiv.org/abs/1910.07207)", "",
    ])
    return "\n".join(lines)


def plot_summary(path):
    import matplotlib.pyplot as plt
    figure, axes = plt.subplots(1, 2, figsize=(10.5, 4), layout="constrained")
    models = ["ridge", "selected"]
    for ax, field, title, unit in zip(axes, FIELDS_OUT, ("Cumulative delivered energy", "Cumulative charging time"),
                                     ("Wh", "minutes")):
        table = forecast_metrics.loc[forecast_metrics.target.eq(field)]
        for name, color, label in zip(models, ("#476aa1", "#19745e"), ("Notebook 18", "Selected forecast")):
            points = table.loc[table.model.eq(name)].sort_values("horizon_cycles")
            ax.plot(points.horizon_cycles, points.mae, "o-", color=color, label=label)
        ax.set(title=title, xlabel="Forecast horizon (cycles)", ylabel=f"Mean absolute error ({unit})")
        ax.set_xticks((5,10,15))
        ax.grid(alpha=.2)
        ax.legend(fontsize=8)
    figure.savefig(path, dpi=145)
    plt.close(figure)


output.mkdir(parents=True, exist_ok=True)
tables = {
    "forecast_metrics": forecast_metrics, "forecast_predictions": predictions, "forecast_selection": selections,
    "original_worst_cases": worst_cases, "controller_metrics": controller_metrics.reset_index(),
    "controller_comparisons": comparisons, "controller_episodes": controller_paths, "sac_training": training_log,
    "sac_replay_sizes": sac_replay_sizes, "measured_replay": measured_replay, "measured_metrics": measured_metrics,
    "raw_charge_features": raw_shapes,
}
for name, table in tables.items():
    table.to_csv(output / f"{name}.csv", index=False)
config = dict(dataset="10.17632/kw34hhw7xg.3", folds=FOLDS, horizons=[5,10,15], forecast_targets=FORECAST_TARGET.tolist(),
              starts=POLICY_STARTS, disturbance_paths=EVAL_DRAWS, sac_training_paths=TRAIN_DRAWS, seeds=SEEDS,
              sac_hidden_layers=[64,64], sac_updates=SAC_UPDATES, sac_batch_size=SAC_BATCH,
              learning_rate=.0003, target_update=TAU, energy_budget_fraction=ENERGY_TARGET,
              training_entropy_fraction=.7, bootstrap_draws=BOOTSTRAP_DRAWS)
(output / "run_config.json").write_text(json.dumps(config, indent=2) + "\n", encoding="utf-8")
(output / "summary.md").write_text(report_text(), encoding="utf-8")
plot_summary(output / "forecast_comparison.png")
torch.save(weights, output / "sac_actor_weights.pt")
files = [*(f"{name}.csv" for name in tables), "run_config.json", "summary.md", "forecast_comparison.png", "sac_actor_weights.pt"]
archive_path = output.with_suffix(".zip")
with ZipFile(archive_path, "w", ZIP_DEFLATED) as bundle:
    for name in files:
        bundle.write(output / name, arcname=f"{output.name}/{name}")
print(f"Report: {output / 'summary.md'}")
print(f"Download: {archive_path}")

Report: /Users/sakibsmacbookm5/Documents/Python work/research-project/SICIP Final Project/reports/lifeguard_mit_bit_charging_sac_results/summary.md
Download: /Users/sakibsmacbookm5/Documents/Python work/research-project/SICIP Final Project/reports/lifeguard_mit_bit_charging_sac_results.zip


## Optional MLflow export

Set `TRACK_WITH_MLFLOW` to `True` to send the completed report to the local LifeGuard-RL MLflow store. Run this cell after the report export, or leave the setting enabled before running the notebook. The report folder and ZIP already exist when this cell runs. You can also import a downloaded report ZIP on your computer using the instructions in the project README.


In [ ]:
TRACK_WITH_MLFLOW = False

if TRACK_WITH_MLFLOW:
    import subprocess
    import sys

    if not (output / "summary.md").is_file() or not output.with_suffix(".zip").is_file():
        raise FileNotFoundError("Run the report cell to create the report and ZIP before exporting to MLflow.")
    command = [
        sys.executable, "-B", "-m", "tracking", "import",
        "--notebook", "22", "--report", str(output), "--record-execution",
    ]
    outcome = subprocess.run(command, cwd=project, check=False)
    if outcome.returncode:
        print("MLflow export failed. The saved report and ZIP can still be imported locally.")
